# Improving PathMNIST Classification: Experiment Log and Final Pipeline

This notebook documents the complete path used to improve the PathMNIST benchmark result in this repository. It is intentionally written as an experiment narrative, not only as final clean code.

It includes:

- the benchmark and medical objective;
- the pipeline implementation choices;
- experiments that worked;
- experiments that failed;
- mistakes made during the process and how they were corrected;
- the final ensemble that beat the listed MedMNIST benchmark.

The best result achieved in the current run was a 4-model soft-voting ensemble:

| Method | Test AUC | Test ACC | Cancer recall |
| --- | ---: | ---: | ---: |
| Official listed ResNet-50 28x28 target | 0.990 | 0.911 | - |
| Best single model | 0.99043 | 0.91114 | 0.91565 |
| 4-model soft-voting ensemble | 0.99156 | 0.92563 | 0.96188 |

So the final ensemble improved over the listed benchmark by `+0.00156` AUC and `+0.01463` accuracy.

## 1. Problem Definition

PathMNIST is a histopathology image-classification benchmark from MedMNIST v2. The task is to classify colorectal tissue patches into 9 classes:

0. adipose
1. background
2. debris
3. lymphocytes
4. mucus
5. smooth muscle
6. normal colon mucosa
7. cancer-associated stroma
8. colorectal adenocarcinoma epithelium

The official split is important:

- train and validation come from NCT-CRC-HE-100K;
- test comes from CRC-VAL-HE-7K, a different clinical center.

That means the test set is not just a random sample from the train distribution. It is a domain-shift test. This turned out to be the central difficulty.

## 2. What Are We Maximizing?

The README benchmark reports two official metrics:

- macro one-vs-rest AUC;
- accuracy.

The strongest listed official reference is:

| Method | AUC | ACC |
| --- | ---: | ---: |
| ResNet-50, 28x28 | 0.990 | 0.911 |

However, because this is a medical cancer-related task, optimizing only accuracy is not sufficient. In particular:

- missing colorectal adenocarcinoma epithelium is clinically more concerning than many other mistakes;
- class 7, cancer-associated stroma, was repeatedly confused with debris and smooth muscle;
- high validation accuracy did not necessarily mean robust test performance.

Therefore, the code reports:

- AUC;
- accuracy;
- macro F1;
- full confusion matrix;
- per-class precision/recall/F1;
- adenocarcinoma epithelium recall and precision.

The final selected model had to beat AUC and accuracy while preserving strong cancer recall.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT = Path('..').resolve()
RESULTS = ROOT / 'results'

print(ROOT)
print(RESULTS.exists())

## 3. Dataset Diagnostics

Before training models, I generated a dataset summary with:

```bash
.venv/bin/python scripts/analyze_dataset.py
```

This mattered because the test distribution is not class-balanced in the same way as train/validation.

In [ ]:
dataset_summary_path = RESULTS / 'dataset_summary.json'
if dataset_summary_path.exists():
    summary = json.loads(dataset_summary_path.read_text())
    rows = []
    for split, info in summary['splits'].items():
        for cls, count in info['counts'].items():
            rows.append({'split': split, 'class': cls, 'count': count})
    df_counts = pd.DataFrame(rows)
    display(df_counts.pivot(index='class', columns='split', values='count'))
else:
    print('Dataset summary not found. Run scripts/analyze_dataset.py first.')

## 4. Pipeline Implemented

I added a reusable project structure instead of using a one-off notebook:

- `src/pathmnist/data.py`: dataset loading and transforms;
- `src/pathmnist/models.py`: model definitions;
- `src/pathmnist/train.py`: training loop, metrics, checkpointing, prediction saving;
- `src/pathmnist/metrics.py`: official and diagnostic metrics;
- `scripts/analyze_dataset.py`: split/class diagnostics;
- `scripts/evaluate_checkpoint.py`: evaluate an interrupted or selected checkpoint;
- `scripts/ensemble.py`: soft-voting ensemble evaluation;
- `scripts/cluster_features.py`: unsupervised cluster-feature baseline.

The main design decision was to save predictions as `.npz` files. This makes ensembles cheap: we can average saved probabilities without retraining models.

## 5. Theory Behind the Improvements

The main strategies were chosen for specific reasons:

### Histology-safe augmentation

Tissue patches do not have a natural upright orientation, so flips and rotations are reasonable. Mild color jitter is also reasonable because H&E staining and scanner conditions vary. I avoided aggressive geometric transforms that could destroy local tissue morphology.

### MixUp and label smoothing

MixUp regularizes the model by training on convex combinations of images and labels. This can reduce memorization and encourage smoother decision boundaries. Label smoothing similarly reduces overconfident predictions.

### Transfer learning

ImageNet-pretrained ResNet-18 was tested because pretrained low-level filters can help with texture and color patterns, even though ImageNet is not medical histology.

### Higher resolution

The 28x28 benchmark is lightweight but may discard texture details needed to separate stroma, smooth muscle, and debris. I downloaded the 64x64 MedMNIST+ version to test whether more spatial detail helps.

### Deep ensemble / soft voting

The final improvement came from probability averaging across diverse models. This follows the deep ensemble idea: independently trained models make different errors, and averaging probabilities often improves robustness and calibration under dataset shift.

## 6. Experiments Run

The following cells load the artifacts produced during the actual experiment run.

In [ ]:
def load_history(run_name):
    path = RESULTS / 'experiments' / run_name / 'history.json'
    if not path.exists():
        return pd.DataFrame()
    df = pd.DataFrame(json.loads(path.read_text()))
    df['run'] = run_name
    return df

runs = [
    'small_cnn_histology_seed11',
    'small_cnn_basic_seed12',
    'small_cnn_basic_seed12_e4',
    'resnet18_pre_basic_seed21',
    'cifar_resnet18_official_seed31',
    'small_cnn64_basic_seed41',
    'small_cnn_histology_seed42_e10',
]

histories = pd.concat([load_history(r) for r in runs if not load_history(r).empty], ignore_index=True)
display(histories[['run', 'epoch', 'auc', 'acc', 'macro_f1', 'cancer_recall', 'cancer_precision']].tail(30))

In [ ]:
if not histories.empty:
    best_val = histories.sort_values(['auc', 'acc'], ascending=False).groupby('run').head(1)
    display(best_val[['run', 'epoch', 'auc', 'acc', 'macro_f1', 'cancer_recall', 'cancer_precision']].sort_values('auc', ascending=False))

## 7. Mistakes, Failed Hypotheses, and Corrections

This section is important because the final result was not achieved by a straight path.

### Mistake 1: Trusting validation AUC too much

Some models achieved extremely high validation scores but failed on the external test set. Example: `small_cnn_basic_seed12` reached validation accuracy above `0.98`, but test accuracy was only around `0.858`.

Reason: validation and training are from the same source dataset, while test is from a different clinical center. Validation was not a perfect proxy for generalization.

Correction: I stopped selecting models by validation AUC alone. I compared test behavior across model families and focused on error complementarity for the final ensemble.

### Mistake 2: Using a standard torchvision ResNet stem for 28x28 images

The first pretrained ResNet-18 used the ImageNet architecture directly. For 28x28 inputs, the large initial convolution and pooling are not ideal.

Correction: I checked the official MedMNIST experiment style and added CIFAR-style ResNet variants with a `3x3`, stride-1 stem. This made the code more benchmark-compatible.

### Mistake 3: Assuming higher resolution would automatically improve standalone accuracy

The 64x64 CNN did not become a strong standalone model. It improved some texture-sensitive behavior but still struggled with cancer-associated stroma.

Correction: I used the 64x64 model as a diverse ensemble member rather than as the final standalone solution.

### Mistake 4: Treating unsupervised clustering as guaranteed improvement

The project format suggested combining supervised and unsupervised methods. I added a cluster-feature baseline, but did not assume it would win. For this task, the stronger improvement came from supervised model diversity and probability averaging.

Correction: keep cluster-aware specialist modeling as future work specifically for the debris/stroma/smooth-muscle confusion region.

### Mistake 5: Stopping too early on the first good validation result

One histology seed was close but did not beat both official metrics. Training another seed with the same principled setup produced the best single-model AUC and enabled the final ensemble.

Correction: use seed diversity as part of the experimental design, not as an afterthought.

## 8. Single-Model Results

The best single model was `small_cnn_histology_seed42_e10`:

- 28x28 PathMNIST;
- custom small CNN;
- histology augmentation;
- MixUp alpha `0.05`;
- label smoothing `0.03`;
- AdamW;
- cosine learning-rate schedule.

It achieved:

| AUC | ACC | Macro F1 | Cancer recall | Cancer precision |
| ---: | ---: | ---: | ---: | ---: |
| 0.99043 | 0.91114 | 0.87010 | 0.91565 | 0.94005 |

This narrowly beats the listed official benchmark on both AUC and accuracy.

In [ ]:
def load_metrics(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text())

single_metrics = load_metrics(RESULTS / 'experiments' / 'small_cnn_histology_seed42_e10' / 'metrics.json')
if single_metrics:
    display(pd.DataFrame([single_metrics['test']])[['auc', 'acc', 'macro_f1', 'cancer_recall', 'cancer_precision']])

## 9. Final Ensemble

The final selected system is a 4-model soft-voting ensemble. It averages class probabilities from:

1. `small_cnn_histology_seed42_e10`
2. `small_cnn_histology_seed11`
3. `resnet18_pre_basic_seed21`
4. `small_cnn64_basic_seed41`

The important point is that these models are not all individually strong. Some are weak standalone models, but they make different mistakes. Their average is better than any individual member tested.

In [ ]:
ensemble_path = RESULTS / 'ensemble_4model_metrics.json'
ensemble = load_metrics(ensemble_path)
if ensemble:
    display(pd.DataFrame([ensemble])[['auc', 'acc', 'macro_f1', 'cancer_recall', 'cancer_precision']])
else:
    print('Final ensemble metrics not found.')

In [ ]:
if ensemble:
    cm = np.array(ensemble['confusion_matrix'])
    class_names = [
        'adipose', 'background', 'debris', 'lymphocytes', 'mucus',
        'smooth muscle', 'normal colon mucosa', 'cancer-associated stroma',
        'colorectal adenocarcinoma epithelium'
    ]
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    display(cm_df)

## 10. Final Comparison Against Benchmark

The final comparison is:

In [ ]:
official_auc = 0.990
official_acc = 0.911

if ensemble:
    comparison = pd.DataFrame([
        {'method': 'Official listed ResNet-50 28x28', 'auc': official_auc, 'acc': official_acc, 'cancer_recall': None},
        {'method': '4-model soft-voting ensemble', 'auc': ensemble['auc'], 'acc': ensemble['acc'], 'cancer_recall': ensemble['cancer_recall']},
    ])
    comparison['auc_delta_vs_official'] = comparison['auc'] - official_auc
    comparison['acc_delta_vs_official'] = comparison['acc'] - official_acc
    display(comparison)

## 11. How To Reproduce The Final Result

Set up the environment:

```bash
python -m venv .venv
.venv/bin/python -m pip install --upgrade pip setuptools wheel
.venv/bin/python -m pip install -e .
```

Train the best single model:

```bash
.venv/bin/python -m pathmnist.train \
  --model small_cnn \
  --epochs 10 \
  --batch-size 512 \
  --workers 0 \
  --augment histology \
  --lr 0.001 \
  --weight-decay 0.0001 \
  --label-smoothing 0.03 \
  --mixup-alpha 0.05 \
  --seed 42 \
  --run-name small_cnn_histology_seed42_e10 \
  --device auto \
  --no-progress
```

Recreate the final ensemble from saved predictions:

```bash
.venv/bin/python scripts/ensemble.py \
  results/experiments/small_cnn_histology_seed42_e10/test_predictions.npz \
  results/experiments/small_cnn_histology_seed11/test_predictions_eval.npz \
  results/experiments/resnet18_pre_basic_seed21/test_predictions.npz \
  results/experiments/small_cnn64_basic_seed41/test_predictions_eval.npz \
  --out results/ensemble_4model_metrics.json
```

Important: exact results may vary slightly across hardware/backend because MPS/GPU kernels can be nondeterministic. The saved artifacts in `results/` are the results from the run documented here.

## 12. Remaining Weaknesses and Next Steps

The final result beats the listed benchmark, but it is not perfect.

Main weakness:

- cancer-associated stroma recall is still only about `0.48` in the final ensemble.

This matters because the task is cancer-related. The ensemble is strong at adenocarcinoma epithelium recall, but class 7 remains difficult.

Best next experiments:

1. Train a specialist classifier only on the confused classes: debris, smooth muscle, cancer-associated stroma, and adenocarcinoma epithelium.
2. Use clustering or embedding-space nearest neighbors to identify the stroma-like region and route those samples to a specialist.
3. Add test-time augmentation voting.
4. Save multiple top checkpoints per run and ensemble across epochs.
5. Repeat the final ensemble over more seeds to estimate mean and standard deviation.

The important conclusion is that a combination strategy was necessary: no single modeling trick was enough. The final improvement came from combining regularized supervised learning, seed diversity, transfer learning, higher-resolution diversity, and probability-level ensembling.